# Day 8: Human in the Loop

Would you let an AI send emails to your manager without reading them first? Transfer money from your account without asking? Probably not. And you shouldn't.

Real AI systems often need a person to step in at the right moment: approve an action, fix a draft, or answer a question the AI can't. Today we learn how to **pause** a graph, wait for a human, and then **continue** from exactly where it stopped.

The analogy I like here is a bank cheque that needs two signatures. The clerk prepares everything, then stops and waits for the manager to sign. The manager might take five minutes or two days. When they sign, the clerk carries on from that exact point, they don't start the paperwork again.

This only works because of what we learned yesterday. The checkpointer saves the state, so the graph can stop, wait as long as needed, and pick up again later.

We'll also try something that feels a bit like magic: going **back in time** to an earlier step, changing something, and replaying from there.

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. `interrupt()`: pause and ask

Inside any node you can call `interrupt(question)`. Here's what happens:

1. The graph **stops** right there, and the checkpointer saves everything.
2. `invoke()` returns. Your question shows up in the result under the key `__interrupt__`.
3. Later, you continue with `invoke(Command(resume=answer), config)`, using the **same thread**.
4. Inside the node, `interrupt(...)` now **returns the answer**, and the node carries on.

Let's start without any LLM, just to see the mechanics clearly.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

In [ ]:
class GreetState(TypedDict):
    name: str
    greeting: str

In [ ]:
def ask_name(state: GreetState) -> dict:
    name = interrupt("What is your name?")     # the graph pauses on this line
    return {"name": name}                      # and this runs after we resume

In [ ]:
def greet(state: GreetState) -> dict:
    return {"greeting": f"Namaste, {state['name']}!"}

In [ ]:
builder = StateGraph(GreetState)

In [ ]:
builder.add_sequence([ask_name, greet])

In [ ]:
builder.add_edge(START, "ask_name")
builder.add_edge("greet", END)

In [ ]:
# A checkpointer is required, otherwise there's nowhere to save the paused state
greet_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "greet-1"}}

result = greet_app.invoke({}, config)

print("The graph paused and asked:", result["__interrupt__"][0].value)
print("Waiting to run:", greet_app.get_state(config).next)

The graph is now sitting there, paused, waiting for us. Let's answer.

In [ ]:
result = greet_app.invoke(Command(resume="Sneha"), config)
print(result["greeting"])

### Something important about resuming

When you resume, the paused node runs **again from its first line**. This time, when it reaches `interrupt()`, it doesn't pause; it returns your answer instead.

### Predict before you run

The node below prints a line *before* calling `interrupt()`. We'll run it once (it pauses), then resume once. How many times will "Preparing the question..." be printed in total?

<details>
<summary>Click to see the answer</summary>

Twice. Once on the first run, and once again when we resume, because the node starts from the top.

That's harmless for a print. But imagine that line was "send an SMS" or "charge the card". It would happen twice. So: **never put side effects before `interrupt()` in the same node.** Put them in a separate node that runs after.
</details>

In [ ]:
def ask_city(state: dict) -> dict:
    print("Preparing the question...")         # this runs again on resume
    city = interrupt("Which city are you from?")
    return {"city": city}

In [ ]:
class CityState(TypedDict):
    city: str

In [ ]:
builder = StateGraph(CityState)

In [ ]:
builder.add_node("ask_city", ask_city)

In [ ]:
builder.add_edge(START, "ask_city")
builder.add_edge("ask_city", END)

In [ ]:
city_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
city_config = {"configurable": {"thread_id": "city-1"}}

In [ ]:
city_app.invoke({}, city_config)
print("--- now resuming ---")
print(city_app.invoke(Command(resume="Nagpur"), city_config))

## 2. Approve or reject

Now a real use case. The AI drafts an email, and a person must approve it before it's "sent". Depending on the answer, we go to `send_email` or `cancel`. We'll use `Command(goto=...)` from Day 3 to choose the next node.

```
  START -> write_email -> human_approval --yes--> send_email -> END
                                  |
                                  +------no-----> cancel -----> END
```

In [ ]:
from typing import Literal

In [ ]:
class EmailState(TypedDict):
    request: str
    draft: str
    status: str

In [ ]:
def write_email(state: EmailState) -> dict:
    draft = llm.invoke(f"Write a short, polite email (under 60 words) to: {state['request']}").content
    return {"draft": draft}

In [ ]:
def human_approval(state: EmailState) -> Command[Literal["send_email", "cancel"]]:
    # We can pass a dictionary to interrupt, so the person sees the draft too
    answer = interrupt({
        "question": "Should I send this email? (yes/no)",
        "draft": state["draft"],
    })

    if answer.strip().lower() in ("yes", "y"):
        return Command(goto="send_email")
    return Command(goto="cancel")

In [ ]:
def send_email(state: EmailState) -> dict:
    # In a real app this is where you'd call your email service
    print("(pretend) Sending the email now...")
    return {"status": "sent"}

In [ ]:
def cancel(state: EmailState) -> dict:
    return {"status": "cancelled"}

In [ ]:
builder = StateGraph(EmailState)

In [ ]:
builder.add_node("write_email", write_email)
builder.add_node("human_approval", human_approval)
builder.add_node("send_email", send_email)
builder.add_node("cancel", cancel)

In [ ]:
builder.add_edge(START, "write_email")
builder.add_edge("write_email", "human_approval")
builder.add_edge("send_email", END)
builder.add_edge("cancel", END)

In [ ]:
email_app = builder.compile(checkpointer=InMemorySaver())
show_graph(email_app)

In [ ]:
email_config = {"configurable": {"thread_id": "email-1"}}

paused = email_app.invoke({"request": "ask my manager for leave on Friday"}, email_config)

question = paused["__interrupt__"][0].value
print(question["question"])
print()
print(question["draft"])

### Your turn

Run the next cell as it is. Then change `"yes"` to `"no"`, change the thread id to `"email-2"`, run the cell above with that new thread, and run this one again. You should see "cancelled".

In [ ]:
my_answer = "yes"      # try "no"

final = email_app.invoke(Command(resume=my_answer), email_config)
print("Status:", final["status"])

## 3. Letting the person edit the AI's work

Approval isn't just yes or no. Often the most useful thing is to let someone **fix** the draft. Whatever you pass in `Command(resume=...)` is exactly what `interrupt()` returns, so the person can send back new text.

In [ ]:
class PostState(TypedDict):
    topic: str
    post: str

In [ ]:
def draft_post(state: PostState) -> dict:
    post = llm.invoke(f"Write a 2-line LinkedIn post about {state['topic']}.").content
    return {"post": post}

In [ ]:
def human_edit(state: PostState) -> dict:
    edited = interrupt({
        "instruction": "Edit the post, or send it back unchanged",
        "post": state["post"],
    })
    return {"post": edited}

In [ ]:
def publish(state: PostState) -> dict:
    print("Publishing:\n", state["post"])
    return {}

In [ ]:
builder = StateGraph(PostState)

In [ ]:
builder.add_sequence([draft_post, human_edit, publish])

In [ ]:
builder.add_edge(START, "draft_post")
builder.add_edge("publish", END)

In [ ]:
post_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
post_config = {"configurable": {"thread_id": "post-1"}}

In [ ]:
paused = post_app.invoke({"topic": "finishing a LangGraph course"}, post_config)
print("The AI wrote:\n", paused["__interrupt__"][0].value["post"])

In [ ]:
my_version = "Day 8 of my LangGraph course done. My agents now ask before they act. #LangGraph"

result = post_app.invoke(Command(resume=my_version), post_config)

## 4. Approving tool calls in an agent

On Day 6 our agent could call any tool freely. For risky tools, like transferring money or deleting files, you want a human to approve each call.

The simplest way is to call `interrupt()` **inside the tool itself**. The agent decides to use the tool, the tool pauses and asks, and only continues if the person says yes.

In [ ]:
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
@tool
def transfer_money(to: str, amount: float) -> str:
    """Transfer money, in rupees, to a person."""
    decision = interrupt(f"Approve a transfer of Rs. {amount} to {to}? (yes/no)")
    if decision.lower() != "yes":
        return "The user cancelled this transfer."
    return f"Transferred Rs. {amount} to {to}."

In [ ]:
bank_tools = [transfer_money]
bank_llm = llm.bind_tools(bank_tools)

In [ ]:
def bank_agent(state: MessagesState) -> dict:
    return {"messages": [bank_llm.invoke(state["messages"])]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("agent", bank_agent)
builder.add_node("tools", ToolNode(bank_tools))

In [ ]:
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

In [ ]:
bank_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
bank_config = {"configurable": {"thread_id": "bank-1"}}

In [ ]:
paused = bank_app.invoke({"messages": [("user", "Send 500 rupees to Kiran")]}, bank_config)
print(paused["__interrupt__"][0].value)

In [ ]:
done = bank_app.invoke(Command(resume="yes"), bank_config)
print(done["messages"][-1].content)

## 5. Asking again until the answer makes sense

People type all sorts of things. If you ask for an age and get "twenty", you want to ask again, nicely, rather than crash. Put the `interrupt()` in a loop inside the node, and keep asking until the answer is valid.

In [ ]:
class AgeState(TypedDict):
    age: int

In [ ]:
def ask_age(state: AgeState) -> dict:
    question = "How old are you?"
    while True:
        answer = interrupt(question)
        if str(answer).isdigit() and 0 < int(answer) < 120:
            return {"age": int(answer)}
        # Not valid, so change the question and loop round to ask again
        question = f"'{answer}' doesn't look like an age. Please type a number:"

In [ ]:
builder = StateGraph(AgeState)

In [ ]:
builder.add_node("ask_age", ask_age)

In [ ]:
builder.add_edge(START, "ask_age")
builder.add_edge("ask_age", END)

In [ ]:
age_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
age_config = {"configurable": {"thread_id": "age-1"}}

### Predict before you run

We'll answer with "twenty", then "-5", then "29". How many times will the graph pause, and what's the final state?

<details>
<summary>Click to see the answer</summary>

It pauses three times (once at the start, and after each of the two bad answers), then finishes after "29". The final state is `{'age': 29}`.

Note that "-5" fails because `"-5".isdigit()` is `False`.
</details>

In [ ]:
result = age_app.invoke({}, age_config)

for reply in ["twenty", "-5", "29"]:
    print("Asked:", result["__interrupt__"][0].value)
    print("Human says:", reply)
    result = age_app.invoke(Command(resume=reply), age_config)
    print()

print("Final:", result)

## 6. Editing the saved state directly

Sometimes an admin needs to fix something by hand, without any node running. `update_state` lets you change a thread's saved state. The update goes through the reducers, exactly as if a node had returned it.

In [ ]:
admin_config = {"configurable": {"thread_id": "greet-2"}}

greet_app.invoke({}, admin_config)                                # pauses at ask_name
greet_app.update_state(admin_config, {"name": "Set by admin"})    # change the saved state

print(greet_app.get_state(admin_config).values)

There's also a debugging trick called a **static breakpoint**. Instead of putting `interrupt()` in your code, you tell `compile` to pause before a node: `interrupt_before=["node_name"]`. It's handy for checking the state just before a tricky step. You continue with `invoke(None, config)`.

In [ ]:
builder = StateGraph(GreetState)

In [ ]:
builder.add_node("greet", greet)

In [ ]:
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

In [ ]:
debug_app = builder.compile(checkpointer=InMemorySaver(), interrupt_before=["greet"])

In [ ]:
debug_config = {"configurable": {"thread_id": "debug-1"}}

In [ ]:
debug_app.invoke({"name": "Vikram"}, debug_config)
print("Paused before:", debug_app.get_state(debug_config).next)

In [ ]:
print(debug_app.invoke(None, debug_config))      # None means "just carry on"

## 7. Time travel

Since the checkpointer saves every step, you can go back to **any** earlier step. Two things you can do from there:

- **Replay**: run again from that point, exactly as it was.
- **Fork**: change something at that point, then run from there. Your original history stays untouched; you get a new branch, like "what if I had chosen differently?"

Example: a joke generator. We'll go back to just before the joke was written, change the topic, and get a new joke, without starting over.

In [ ]:
class JokeState(TypedDict):
    topic: str
    joke: str

In [ ]:
def write_joke(state: JokeState) -> dict:
    return {"joke": llm.invoke(f"Tell a one-line clean joke about {state['topic']}.").content}

In [ ]:
builder = StateGraph(JokeState)

In [ ]:
builder.add_node("write_joke", write_joke)

In [ ]:
builder.add_edge(START, "write_joke")
builder.add_edge("write_joke", END)

In [ ]:
joke_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
joke_config = {"configurable": {"thread_id": "jokes"}}

In [ ]:
original = joke_app.invoke({"topic": "programmers"}, joke_config)
print("Original joke:", original["joke"])

First, let's look at the history and find the checkpoint where `write_joke` was about to run.

In [ ]:
history = list(joke_app.get_state_history(joke_config))

for snap in history:
    print(f"step {snap.metadata['step']:>2} | next: {snap.next} | topic: {snap.values.get('topic')}")

In [ ]:
before_joke = next(snap for snap in history if snap.next == ("write_joke",))

Now we fork: change the topic at that checkpoint, then continue from there.

In [ ]:
fork_config = joke_app.update_state(before_joke.config, {"topic": "cats"})

forked = joke_app.invoke(None, fork_config)
print("Forked joke:", forked["joke"])

In [ ]:
print("Checkpoints in this thread now:", len(list(joke_app.get_state_history(joke_config))))

## 8. Common mistakes

**Mistake 1: Using `interrupt()` without a checkpointer.**
This one is confusing the first time. The graph *does* pause and show you the question. But when you try to resume, there's no saved state to resume from, so it fails.

In [ ]:
builder = StateGraph(GreetState)

In [ ]:
builder.add_sequence([ask_name, greet])

In [ ]:
builder.add_edge(START, "ask_name")
builder.add_edge("greet", END)

In [ ]:
no_memory_app = builder.compile()        # no checkpointer

In [ ]:
paused = no_memory_app.invoke({})
print("It paused fine:", paused["__interrupt__"][0].value)

try:
    no_memory_app.invoke(Command(resume="Sneha"))
except RuntimeError as error:
    print("But resuming fails:", error)

**Fix:** compile with `checkpointer=InMemorySaver()` (or SQLite / Postgres) and pass a `thread_id`.

**Mistake 2: Resuming on a different thread.**
If you paused on `"email-1"` and resume with `"email-2"`, LangGraph has nothing paused on `"email-2"`, so your answer goes nowhere. Keep the config from the first call and reuse it.

**Mistake 3: Side effects before `interrupt()`.**
You saw this in section 1. The node reruns from the top on resume, so anything before the `interrupt()` happens twice. Move payments, emails and database writes into a node *after* the approval.

**Mistake 4: Resuming with plain input instead of `Command(resume=...)`.**
`email_app.invoke("yes", config)` does not answer the question. It's treated as new input. Always wrap the answer: `Command(resume="yes")`.

## 9. Practice

**Exercise 1.** Build `START -> plan_trip -> human_review -> END`. `plan_trip` asks the LLM for a short one-day plan for a `city`. In `human_review`, the person answers either `"ok"` (keep the plan as is) or some feedback text; if it's feedback, store it in a `feedback` key.

**Exercise 2.** Ask for two numbers using two `interrupt()` calls in **two different nodes**, then add them in a third node. You'll need to resume twice.

In [ ]:
# Solution 1
class TripState(TypedDict):
    city: str
    plan: str
    feedback: str

In [ ]:
def plan_trip(state: TripState) -> dict:
    plan = llm.invoke(f"Give a 3-item one-day plan for {state['city']}. Keep it very short.").content
    return {"plan": plan}

In [ ]:
def human_review(state: TripState) -> dict:
    answer = interrupt({"plan": state["plan"], "ask": "Reply 'ok' or give feedback"})
    if answer == "ok":
        return {}
    return {"feedback": answer}

In [ ]:
builder = StateGraph(TripState)

In [ ]:
builder.add_sequence([plan_trip, human_review])

In [ ]:
builder.add_edge(START, "plan_trip")
builder.add_edge("human_review", END)

In [ ]:
trip_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
trip_config = {"configurable": {"thread_id": "trip"}}

trip_app.invoke({"city": "Jaipur"}, trip_config)
print(trip_app.invoke(Command(resume="Please add a food stop"), trip_config))

In [ ]:
# Solution 2
class SumState(TypedDict):
    a: int
    b: int
    total: int

In [ ]:
def ask_first(state: SumState) -> dict:
    return {"a": int(interrupt("First number?"))}

In [ ]:
def ask_second(state: SumState) -> dict:
    return {"b": int(interrupt("Second number?"))}

In [ ]:
def add_numbers(state: SumState) -> dict:
    return {"total": state["a"] + state["b"]}

In [ ]:
builder = StateGraph(SumState)

In [ ]:
builder.add_sequence([ask_first, ask_second, add_numbers])

In [ ]:
builder.add_edge(START, "ask_first")
builder.add_edge("add_numbers", END)

In [ ]:
sum_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
sum_config = {"configurable": {"thread_id": "sum"}}

In [ ]:
sum_app.invoke({}, sum_config)                        # pauses at the first question
sum_app.invoke(Command(resume="7"), sum_config)       # pauses at the second question
result = sum_app.invoke(Command(resume="35"), sum_config)

print(result)
assert result["total"] == 42

## Wrapping up

`interrupt(value)` pauses the graph and hands `value` to whoever is watching. `invoke(Command(resume=answer), config)` continues, and `interrupt()` returns the answer. This needs a checkpointer and the same thread id.

Remember that the paused node reruns from the top when you resume, so keep side effects after the approval. You can use this for approvals, edits, approving tool calls and validating input. And because every step is saved, you can inspect, edit (`update_state`) and even fork the past.

Tomorrow is about user experience: **streaming**, so people see answers appear word by word and get live progress updates instead of staring at a spinner.